# 🎬 Notebook 1: Movie Ticket Booking — Class Design

In this notebook we design the **objects** (classes) we'll need for a movie-ticket booking system — the same kind of system behind BookMyShow, Fandango, or AMC online.

We'll answer these beginner questions first:

- **What are the "things" (nouns) in the problem?** Those become classes.
- **What can each thing *do* (verbs)?** Those become methods.
- **How do the things relate?** (a cinema *has* screens, a show *has* seats …)

We won't implement the booking logic yet — that's notebook 2. Here we just build the skeleton and play with it.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/movie-ticket-booking
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Requirements (plain English first!)

> Every OOD problem starts by writing requirements in plain English. If you can't say it in English, you can't write it in code.

A **cinema** (like "AMC Downtown") has several **screens** (screen 1, screen 2, …).
Each screen runs **shows** (a *movie* at a specific *start time*, e.g. *Inception at 7 pm*).
A show has a grid of **seats** (A1, A2, …). A **user** picks seats and creates a **booking**.
A booking needs a **payment**. Two users must never be allowed to book the same seat.

### Nouns → candidate classes
`Movie`, `Cinema`, `Screen`, `Show`, `Seat`, `User`, `Booking`, `Payment`

### Verbs → candidate methods
`search shows`, `pick seats`, `hold seat`, `book`, `pay`, `cancel`

### Picture (simple UML)

```
Cinema  1..*-->  Screen  1..*-->  Show  1..*-->  Seat
                                    ^
                                    | booked for
                                Booking  <-- User
                                    |
                                    v
                                Payment
```


## 2. First attempt: the "bad" way — one giant class 🧟

Beginners often stuff everything into one class. Let's show that first so we can *feel* the pain before we fix it.


In [ ]:
# BAD: one "god class" that knows about movies, seats, money, users -- everything.
class MovieSystem:
    def __init__(self):
        self.data = {}         # messy dict of everything
    def book(self, user, movie, time, seats, card_number):
        # 50 lines of tangled logic would go here:
        # - find the movie
        # - check seats
        # - charge the card
        # - store the booking
        # - send email
        ...

# Problems:
# 1. Impossible to test one piece (e.g. "can I just test seat logic?").
# 2. Impossible to reuse (e.g. what if we add a Concert system later?).
# 3. Any change risks breaking everything -- the class is a spaghetti bowl.
print("MovieSystem defined -- but we won't use it. It's the anti-pattern.")


## 3. Better: split into small, single-responsibility classes ✅

Each class does **one thing well** (Single Responsibility Principle — the "S" in SOLID).
We use Python's `@dataclass` because it removes boilerplate (`__init__`, `__repr__`, …).


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Optional

# --- Value-ish objects (just data) ---------------------------------
@dataclass(frozen=True)          # frozen = immutable, safer to share
class Movie:
    title: str
    duration_min: int            # e.g. 148 for Inception

@dataclass(frozen=True)
class User:
    id: int
    name: str
    email: str

# --- Seat with an explicit status (a tiny state machine) -----------
class SeatStatus(Enum):
    FREE = "free"        # nobody has touched it
    HELD = "held"        # in someone's cart (temporary)
    BOOKED = "booked"    # paid for

@dataclass
class Seat:
    row: str             # "A", "B", ...
    number: int          # 1, 2, ...
    price: float = 12.5  # dollars
    status: SeatStatus = SeatStatus.FREE

    @property
    def id(self) -> str:
        return f"{self.row}{self.number}"   # "A1", "B7" ...

# --- The "container" objects ---------------------------------------
@dataclass
class Screen:
    id: int
    name: str            # "Screen 1"

@dataclass
class Show:
    id: int
    movie: Movie
    screen: Screen
    start: datetime
    seats: dict = field(default_factory=dict)  # "A1" -> Seat

    def add_seat(self, seat: Seat):
        self.seats[seat.id] = seat

@dataclass
class Cinema:
    name: str
    screens: list = field(default_factory=list)
    shows: list = field(default_factory=list)

# --- Booking + Payment ---------------------------------------------
class PaymentStatus(Enum):
    PENDING = "pending"; PAID = "paid"; FAILED = "failed"

@dataclass
class Payment:
    id: int
    amount: float
    status: PaymentStatus = PaymentStatus.PENDING

class BookingStatus(Enum):
    PENDING = "pending"; CONFIRMED = "confirmed"; CANCELLED = "cancelled"

@dataclass
class Booking:
    id: int
    user: User
    show: Show
    seats: list                     # list[Seat]
    status: BookingStatus = BookingStatus.PENDING
    payment: Optional[Payment] = None

    @property
    def total(self) -> float:
        return sum(s.price for s in self.seats)

print("All classes defined. Each has ONE job.")


### Why split it like this?
- **`Seat` doesn't know what a `User` is.** If we build a concert-hall system tomorrow we can reuse `Seat`.
- **`Payment` is separate** so we can swap Stripe for PayPal without touching `Booking`.
- **`Show` owns its seats** — a natural "aggregate root" in Domain-Driven Design terms.
- Every class is tiny, so each is easy to **read, test, and change**.


## 4. Let's actually build a little cinema and print it out

In [ ]:
from datetime import datetime

# Build a cinema with 1 screen and 1 show of 10 seats (rows A & B, 1..5)
amc      = Cinema(name="AMC Downtown")
screen1  = Screen(id=1, name="Screen 1")
amc.screens.append(screen1)

inception = Movie(title="Inception", duration_min=148)
show1 = Show(id=101, movie=inception, screen=screen1,
             start=datetime(2030, 1, 1, 19, 0))
for row in "AB":
    for n in range(1, 6):
        show1.add_seat(Seat(row=row, number=n))
amc.shows.append(show1)

# Pretty-print a seat map.  [ ]=FREE  [H]=HELD  [X]=BOOKED
def seat_map(show):
    rows = sorted({s.row for s in show.seats.values()})
    for r in rows:
        line = f" {r} "
        for n in sorted(s.number for s in show.seats.values() if s.row == r):
            s = show.seats[f"{r}{n}"]
            line += {SeatStatus.FREE: "[ ]",
                     SeatStatus.HELD: "[H]",
                     SeatStatus.BOOKED: "[X]"}[s.status]
        print(line)
    print("    " + "".join(f" {n} " for n in range(1, 6)))

print(f"{show1.movie.title} at {show1.start:%H:%M} on {show1.screen.name}\n")
seat_map(show1)


## 5. Tiny exercise — mark a seat as BOOKED and redraw

Before moving on, flip one seat's status by hand so you *see* how the seat map updates. In notebook 2 we'll do this properly (with concurrency).


In [ ]:
show1.seats["A3"].status = SeatStatus.BOOKED
show1.seats["B1"].status = SeatStatus.HELD
seat_map(show1)

# Reset so the notebook is idempotent (safe to re-run top to bottom).
show1.seats["A3"].status = SeatStatus.FREE
show1.seats["B1"].status = SeatStatus.FREE


## 6. What's next?

In **notebook 2** we implement `BookingService` and learn:
1. How the "obvious" code is **buggy** when two users click at the same time.
2. How a **lock** fixes the race.
3. How real systems use **timed holds** so seats don't stay stuck forever.
4. How this maps to **DB transactions** or **Redis locks** in production.
